<a href="https://colab.research.google.com/github/MJMortensonWarwick/IB2AD0_Data_Science_GenerativeAI/blob/main/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

### My takeaways

_Two lines in my own words: what clicked, and where I would use it in finance._

- 
- 


# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  0


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 28528.5605
Epoch [10/100], Loss: 30405.0859
Epoch [10/100], Loss: 26416.9941
Epoch [10/100], Loss: 35330.6133
Epoch [10/100], Loss: 29693.7754
Epoch [10/100], Loss: 29840.9043
Epoch [10/100], Loss: 27067.2305
Epoch [10/100], Loss: 23869.3906
Epoch [20/100], Loss: 28366.0879
Epoch [20/100], Loss: 26476.957
Epoch [20/100], Loss: 30901.2402
Epoch [20/100], Loss: 29061.3516
Epoch [20/100], Loss: 31138.6309
Epoch [20/100], Loss: 34577.7305
Epoch [20/100], Loss: 27083.3945
Epoch [20/100], Loss: 12501.0898
Epoch [30/100], Loss: 28213.918
Epoch [30/100], Loss: 26338.9277
Epoch [30/100], Loss: 29804.2617
Epoch [30/100], Loss: 37910.8594
Epoch [30/100], Loss: 25209.5508
Epoch [30/100], Loss: 28845.7441
Epoch [30/100], Loss: 28115.0566
Epoch [30/100], Loss: 58824.9258
Epoch [40/100], Loss: 29339.1602
Epoch [40/100], Loss: 25255.0898
Epoch [40/100], Loss: 37390.8203
Epoch [40/100], Loss: 29385.1992
Epoch [40/100], Loss: 28833.375
Epoch [40/100], Loss: 27067.5293
Epoch [40/100

Epoch [50/100], Loss: 34469.5938
Epoch [50/100], Loss: 28265.7852
Epoch [50/100], Loss: 25779.0059
Epoch [50/100], Loss: 30730.332
Epoch [50/100], Loss: 27396.8848
Epoch [50/100], Loss: 27704.5723
Epoch [50/100], Loss: 30010.5547
Epoch [50/100], Loss: 44549.4648
Epoch [60/100], Loss: 28319.2656
Epoch [60/100], Loss: 27964.1973
Epoch [60/100], Loss: 33306.0234
Epoch [60/100], Loss: 30629.957
Epoch [60/100], Loss: 23513.6406
Epoch [60/100], Loss: 32912.25
Epoch [60/100], Loss: 28441.1582
Epoch [60/100], Loss: 22891.9453
Epoch [70/100], Loss: 29928.6602
Epoch [70/100], Loss: 27402.0684
Epoch [70/100], Loss: 33359.5703
Epoch [70/100], Loss: 35872.4844
Epoch [70/100], Loss: 27113.0293
Epoch [70/100], Loss: 18740.4082
Epoch [70/100], Loss: 30497.8223
Epoch [70/100], Loss: 48700.2852
Epoch [80/100], Loss: 27871.1895
Epoch [80/100], Loss: 31393.2559


Epoch [80/100], Loss: 25964.6367
Epoch [80/100], Loss: 34586.3867
Epoch [80/100], Loss: 28626.3496
Epoch [80/100], Loss: 29518.2793
Epoch [80/100], Loss: 25510.0352
Epoch [80/100], Loss: 28026.4141
Epoch [90/100], Loss: 28041.5371
Epoch [90/100], Loss: 34282.1328
Epoch [90/100], Loss: 27334.1406
Epoch [90/100], Loss: 29622.0
Epoch [90/100], Loss: 21918.0254
Epoch [90/100], Loss: 26913.4121
Epoch [90/100], Loss: 34506.8438
Epoch [90/100], Loss: 29754.2754
Epoch [100/100], Loss: 28295.8652
Epoch [100/100], Loss: 28061.373
Epoch [100/100], Loss: 32941.9688
Epoch [100/100], Loss: 25984.3457
Epoch [100/100], Loss: 28609.9316
Epoch [100/100], Loss: 26117.1816
Epoch [100/100], Loss: 32062.6543
Epoch [100/100], Loss: 25616.0371


We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 25356.1220703125


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,2.884748,219.0
1,2.884748,70.0
2,2.884748,202.0
3,2.884748,230.0
4,2.884748,111.0
...,...,...
84,2.884748,153.0
85,2.884748,98.0
86,2.884748,37.0
87,2.884748,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?